# AI-Powered Root Cause Analysis using Groq

This notebook performs automated Root Cause Analysis (RCA) on anomalous logs using a Large Language Model served through the Groq API.

For each detected anomaly, the surrounding context is analysed to determine:

- Probable Root Cause
- Severity
- Impact
- Recommended Fixes (Immediate, Permanent, Preventive)
- Recommendations
- Confidence Score

The generated report can assist Site Reliability Engineers (SREs) in quickly identifying and resolving production issues.

In [8]:
!pip install groq pandas python-dotenv -q


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [9]:
import json
import os
import re
import time
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from groq import Groq

In [10]:
load_dotenv(override=True)

client = Groq(
    api_key=os.getenv("GROQ_API_KEY")
)

# Sanity check
response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[{"role": "user", "content": "Say SUCCESS"}]
)

print(response.choices[0].message.content)

SUCCESS!


# Loading Context

In [11]:
with open("../data/context/anomaly_context.json", "r") as f:
    contexts = json.load(f)

print(f"Loaded {len(contexts)} anomaly contexts.")

Loaded 248 anomaly contexts.


In [12]:
def build_prompt(context):
    return f"""
You are an expert Site Reliability Engineer (SRE), DevOps Engineer, and Production Support Specialist.

Analyze the following anomalous log and perform a comprehensive Root Cause Analysis (RCA).

Return ONLY valid JSON with EXACTLY these PascalCase key names (no snake_case, no lowercase):

{{
    "RootCause": "short description of the root cause",
    "Reason": "why this happened",
    "Severity": "Critical | High | Medium | Low",
    "Impact": "business and operational impact",
    "ImmediateFixes": ["step 1", "step 2", "step 3"],
    "PermanentFixes": ["fix 1", "fix 2", "fix 3"],
    "PreventiveMeasures": ["measure 1", "measure 2", "measure 3"],
    "Recommendations": ["rec 1", "rec 2", "rec 3"],
    "Confidence": "90%"
}}

Current Log:
{context["CurrentLog"]}

Component: {context["Component"]}
Severity: {context["Severity"]}
Contains Error: {context["ContainsError"]}
Authentication Success: {context["AuthenticationSuccess"]}
Anomaly Score: {context["AnomalyScore"]}

Nearby Logs:
{json.dumps(context["NearbyLogs"], indent=4)}

Provide concise, technically accurate, and actionable recommendations suitable for production environments.
"""

## Run RCA on Anomaly Contexts

Processes the top N anomaly contexts through the Groq API.
Includes retry logic for rate limits (429 errors).

In [13]:
TOP_K = 5   # Change this to process more anomalies

rca_results = []

for i, context in enumerate(contexts[:TOP_K]):

    print(f"Processing {i+1}/{TOP_K} (LogIndex: {context['LogIndex']})")

    prompt = build_prompt(context)

    max_retries = 3
    for retry in range(max_retries):
        try:
            response = client.chat.completions.create(
                model="llama-3.1-8b-instant",
                temperature=0,
                response_format={"type": "json_object"},
                messages=[
                    {"role": "user", "content": prompt}
                ]
            )
            rca_results.append({
                "LogIndex":             context["LogIndex"],
                "CurrentLog":           context["CurrentLog"],
                "Component":            context["Component"],
                "OriginalSeverity":     context["Severity"],
                "ContainsError":        context["ContainsError"],
                "AuthenticationSuccess": context["AuthenticationSuccess"],
                "AnomalyScore":         context["AnomalyScore"],
                "NearbyLogs":           context["NearbyLogs"],
                "Response":             response.choices[0].message.content
            })
            print(f"  ✅ Done")
            break
        except Exception as e:
            print(f"  ⚠️  Retry {retry+1}/{max_retries}: {e}")
            if "429" in str(e) or "rate" in str(e).lower():
                time.sleep(20)
            else:
                time.sleep(2)

    time.sleep(1)

print(f"\nCollected {len(rca_results)} raw responses.")

Processing 1/5 (LogIndex: 42)
  ✅ Done
Processing 2/5 (LogIndex: 51)
  ✅ Done
Processing 3/5 (LogIndex: 52)
  ✅ Done
Processing 4/5 (LogIndex: 53)
  ✅ Done
Processing 5/5 (LogIndex: 86)
  ✅ Done

Collected 5 raw responses.


## Parse JSON Responses

The responses returned by the language model are cleaned and converted into structured Python dictionaries. Any malformed responses are skipped gracefully.

In [14]:
# Key normalizer: the model sometimes returns lowercase/snake_case keys.
# This maps any variant to the expected PascalCase keys.
KEY_MAP = {
    "rootcause":          "RootCause",
    "root_cause":         "RootCause",
    "reason":             "Reason",
    "severity":           "Severity",
    "impact":             "Impact",
    "immediatefix":       "ImmediateFixes",
    "immediatefix":       "ImmediateFixes",
    "immediate_fix":      "ImmediateFixes",
    "immediatefixes":     "ImmediateFixes",
    "immediate_fixes":    "ImmediateFixes",
    "permanentfix":       "PermanentFixes",
    "permanent_fix":      "PermanentFixes",
    "permanentfixes":     "PermanentFixes",
    "permanent_fixes":    "PermanentFixes",
    "preventivemeasure":  "PreventiveMeasures",
    "preventive_measure": "PreventiveMeasures",
    "preventivemeasures": "PreventiveMeasures",
    "preventive_measures":"PreventiveMeasures",
    "recommendation":     "Recommendations",
    "recommendations":    "Recommendations",
    "confidence":         "Confidence",
}

def normalize_keys(d):
    """Remap any model-returned key variants to expected PascalCase."""
    out = {}
    for k, v in d.items():
        normalized = KEY_MAP.get(k.lower().replace(' ', '_'), k)
        out[normalized] = v
    return out

parsed_results = []

for item in rca_results:

    try:
        raw = item["Response"].strip()
        # Strip markdown code fences if present
        raw = raw.replace("```json", "").replace("```", "").strip()

        result = json.loads(raw)
        result = normalize_keys(result)

        # Attach context metadata
        result["LogIndex"]              = item["LogIndex"]
        result["CurrentLog"]            = item["CurrentLog"]
        result["Component"]             = item["Component"]
        result["OriginalSeverity"]      = item["OriginalSeverity"]
        result["ContainsError"]         = item["ContainsError"]
        result["AuthenticationSuccess"] = item["AuthenticationSuccess"]
        result["AnomalyScore"]          = item["AnomalyScore"]
        result["NearbyLogs"]            = item["NearbyLogs"]

        # Ensure list fields are always lists
        for list_col in ["ImmediateFixes", "PermanentFixes", "PreventiveMeasures", "Recommendations"]:
            if list_col in result and not isinstance(result[list_col], list):
                result[list_col] = [str(result[list_col])]

        parsed_results.append(result)

    except Exception as e:
        print(f"Skipped Log {item['LogIndex']} : {e}")

print(f"Successfully parsed {len(parsed_results)} results.")

Successfully parsed 5 results.


## Create Structured RCA Report

The parsed JSON responses are converted into a Pandas DataFrame for further analysis and visualization.

In [15]:
rca_df = pd.DataFrame(parsed_results)

print("Columns:", rca_df.columns.tolist())
print("Shape  :", rca_df.shape)

rca_df.head()

Columns: ['RootCause', 'Reason', 'Severity', 'Impact', 'ImmediateFixes', 'PermanentFixes', 'PreventiveMeasures', 'Recommendations', 'Confidence', 'LogIndex', 'CurrentLog', 'Component', 'OriginalSeverity', 'ContainsError', 'AuthenticationSuccess', 'AnomalyScore', 'NearbyLogs']
Shape  : (5, 17)


,RootCause,Reason,Severity,Impact,ImmediateFixes,PermanentFixes,PreventiveMeasures,Recommendations,Confidence,LogIndex,CurrentLog,Component,OriginalSeverity,ContainsError,AuthenticationSuccess,AnomalyScore,NearbyLogs
0,Incorrect message encoding causing SMPP packet...,"The message encoding was set to 0, which is no...",Critical,Potential loss of business-critical SMS messag...,[Set the message encoding to a valid value (e....,[Update the message encoding logic to automati...,[Regularly review and update the message encod...,[Consider implementing a message encoding libr...,95%,42,[Globals.cpp|04136|02:46:17:101|~CR~][77d27ccb...,Globals.cpp,~CR~,False,False,0.914535,[[Globals.cpp|04497|02:46:17:099|~CR~][77d27cc...
1,Unintended SMPP packet insertion into the queu...,The SMPP packet handling mechanism is not prop...,Critical,"Potential data corruption, loss of business-cr...",[Disable the SMPP packet handling mechanism to...,[Implement a robust SMPP packet validation mec...,[Regularly review and update the SMPP packet h...,[Conduct a thorough review of the SMPP packet ...,95%,51,[Globals.cpp|01908|02:46:17:465|~CR~][8afe86c1...,Globals.cpp,~CR~,False,False,0.857280,[[Globals.cpp|04087|02:46:17:112|~CR~]AccountI...
2,Uncontrolled SMS delivery loop due to incorrec...,The system failed to properly handle the RETRY...,Critical,High business and operational impact due to po...,[Implement a retry mechanism with a timeout to...,[Review and refactor the SMS delivery logic to...,[Regularly review and update the system to ens...,[Implement a message queue to handle messages ...,95%,52,[Globals.cpp|01816|02:46:17:496|~CR~][8afe86aa...,Globals.cpp,~CR~,False,False,0.970583,[[Globals.cpp|04124|02:46:17:112|~ER~]Received...
3,Authentication failure due to missing Auth Object,"The Auth Object was not found, causing the aut...",Critical,Authentication failure may lead to loss of ser...,[Check the Auth Object configuration and ensur...,[Implement a fallback mechanism to handle miss...,[Regularly back up the Auth Object configurati...,[Implement a more robust authentication mechan...,95%,53,[Globals.cpp|01908|02:46:17:496|~CR~][8afe86aa...,Globals.cpp,~CR~,False,False,0.769156,[[Globals.cpp|04136|02:46:17:112|~CR~][77d27cd...
4,Insufficient balance for EXPLORE subscription,The user does not have enough balance to subsc...,Critical,"Failed EXPLORE subscription, potential loss of...",[Check user balance before processing EXPLORE ...,[Integrate balance check API with EXPLORE subs...,[Regularly monitor user balances and EXPLORE s...,[Review and update user balance check mechanis...,95%,86,[Globals.cpp|04136|02:46:17:590|~CR~][77d27ce5...,Globals.cpp,~CR~,False,False,0.785040,[[Globals.cpp|04124|02:46:17:589|~ER~]Received...


In [16]:
Path("../outputs").mkdir(exist_ok=True)

rca_df.to_csv(
    "../outputs/root_cause_analysis.csv",
    index=False
)

print("✅ Root Cause Analysis saved to ../outputs/root_cause_analysis.csv")

✅ Root Cause Analysis saved to ../outputs/root_cause_analysis.csv


In [17]:
from IPython.display import display

print("Total Anomalies Analysed :", len(rca_df))

print("\nSeverity Distribution")
display(rca_df["Severity"].value_counts())

print("\nMost Common Root Causes")
display(rca_df["RootCause"].value_counts())

print("\nConfidence Distribution")
display(rca_df["Confidence"].value_counts())

Total Anomalies Analysed : 5

Severity Distribution


Severity
Critical    5
Name: count, dtype: int64


Most Common Root Causes


RootCause
Incorrect message encoding causing SMPP packet corruption                                                 1
Unintended SMPP packet insertion into the queue due to a misconfigured SMPP packet handling mechanism.    1
Uncontrolled SMS delivery loop due to incorrect handling of RETRY_MSG                                     1
Authentication failure due to missing Auth Object                                                         1
Insufficient balance for EXPLORE subscription                                                             1
Name: count, dtype: int64


Confidence Distribution


Confidence
95%    5
Name: count, dtype: int64

## Display Individual Anomaly Report

In [19]:
# from IPython.display import Markdown, display

# for _, row in rca_df.iterrows():

#     display(Markdown(f"""
# ---
# ## 🚨 Anomaly #{row['LogIndex']}

# **Component:** {row.get('Component', 'N/A')}  
# **Anomaly Score:** {row.get('AnomalyScore', 'N/A')}  
# **Original Severity:** {row.get('OriginalSeverity', 'N/A')}

# ### 🔍 Root Cause
# {row['RootCause']}

# ### 📖 Reason
# {row['Reason']}

# ### ⚠️ AI Estimated Severity
# **{row['Severity']}**

# ### 💥 Impact
# {row['Impact']}
# """))

#     print("🚑 Immediate Fixes")
#     for fix in row['ImmediateFixes']:
#         print(f"  • {fix}")

#     print("\n🔧 Permanent Fixes")
#     for fix in row['PermanentFixes']:
#         print(f"  • {fix}")

#     print("\n🛡 Preventive Measures")
#     for m in row['PreventiveMeasures']:
#         print(f"  • {m}")

#     print("\n💡 Recommendations")
#     for rec in row['Recommendations']:
#         print(f"  • {rec}")

#     display(Markdown(f"**🎯 Confidence:** {row['Confidence']}"))

## Look Up a Specific Anomaly by Log Index

In [20]:
def show_anomaly(log_index):
    """Print a full RCA report for the given log index."""
    matches = rca_df[rca_df["LogIndex"] == log_index]
    if matches.empty:
        print(f"No RCA result found for LogIndex={log_index}")
        return

    row = matches.iloc[0]

    print("=" * 80)
    print(f"Anomaly ID        : {row['LogIndex']}")
    print(f"Component         : {row.get('Component', 'N/A')}")
    print(f"Anomaly Score     : {row.get('AnomalyScore', 'N/A')}")
    print(f"Original Severity : {row.get('OriginalSeverity', 'N/A')}")
    print(f"Root Cause        : {row['RootCause']}")
    print(f"Reason            : {row['Reason']}")
    print(f"AI Severity       : {row['Severity']}")
    print(f"Impact            : {row['Impact']}")

    print("\nCurrent Log:")
    print(f"  {row.get('CurrentLog', 'N/A')}")

    print("\n🚑 Immediate Fixes")
    for fix in row["ImmediateFixes"]:
        print("  •", fix)

    print("\n🔧 Permanent Fixes")
    for fix in row["PermanentFixes"]:
        print("  •", fix)

    print("\n🛡 Preventive Measures")
    for fix in row["PreventiveMeasures"]:
        print("  •", fix)

    print("\n💡 Recommendations")
    for rec in row["Recommendations"]:
        print("  •", rec)

    print(f"\n🎯 Confidence : {row['Confidence']}")
    print("=" * 80)

In [21]:
# Show the first analysed anomaly
if not rca_df.empty:
    first_idx = rca_df.iloc[0]["LogIndex"]
    show_anomaly(first_idx)
else:
    print("rca_df is empty — run the RCA loop cells above first.")

Anomaly ID        : 42
Component         : Globals.cpp
Anomaly Score     : 0.91453516
Original Severity : ~CR~
Root Cause        : Incorrect message encoding causing SMPP packet corruption
Reason            : The message encoding was set to 0, which is not a valid encoding for the given message type. This caused the SMPP packet to become corrupted, leading to the anomaly.
AI Severity       : Critical
Impact            : Potential loss of business-critical SMS messages, disruption of service, and revenue loss due to failed message delivery.

Current Log:
  [Globals.cpp|04136|02:46:17:101|~CR~][77d27ccb]Sent Byte to Router:smi: 2010283211 teleservice_id: 4098 originating_address: "15099" destination_address: "255655592461" message_id: 138 message_type: SMS_DELIVER msg_encode: 0 user_data_type: 0 user_data_length: 157 user_data: "ZAWADI KEMKEM! Simujanja, Saajanja, Router na Tablet, bila kusahau pesa taslim hadi Tsh 8,000,000. Tuma neno BOX kwenda 15099 kisha chagua sanduku la bahati." us